# Módulos estándar · Concurrencia

Ejecutar varias tareas a la vez con hilos (`threading`, `queue`, `concurrent.futures`), procesos (`multiprocessing`) y corrutinas (`asyncio`). Los hilos y las corrutinas sirven para tareas que esperan (E/S); para cálculo intensivo hacen falta procesos.

## 1. Primeros hilos

Lanzar tres hilos que simulan tareas de distinta duración con `time.sleep`, esperar a que terminen con `join` y comprobar que el tiempo total es el de la tarea más larga, no la suma.

In [1]:
import threading
import time

def tarea(nombre, segundos):
    time.sleep(segundos)
    print(f"{nombre} termina tras {segundos} s")

inicio = time.perf_counter()
hilos = [threading.Thread(target=tarea, args=(f"tarea {i}", s)) for i, s in enumerate([0.3, 0.1, 0.2], 1)]
for hilo in hilos:
    hilo.start()
for hilo in hilos:
    hilo.join()
print(f"total: {time.perf_counter() - inicio:.2f} s")

tarea 2 termina tras 0.1 s
tarea 3 termina tras 0.2 s
tarea 1 termina tras 0.3 s
total: 0.30 s


## 2. Condición de carrera

Ocho hilos suman 1000 veces cada uno a un contador compartido: leen el valor, ceden el turno y escriben el valor más uno. Como las operaciones de los hilos se intercalan, varios leen el mismo valor y se pierden incrementos.

In [2]:
contador = 0

def incrementa(veces):
    global contador
    for _ in range(veces):
        actual = contador
        time.sleep(0)  # cede el turno entre leer y escribir
        contador = actual + 1

hilos = [threading.Thread(target=incrementa, args=(1000,)) for _ in range(8)]
for hilo in hilos:
    hilo.start()
for hilo in hilos:
    hilo.join()
print(f"esperado 8000, obtenido {contador}")

esperado 8000, obtenido 1287


## 3. Corregir con `Lock`

Proteger la sección crítica (leer y escribir el contador) con un `threading.Lock` para que solo un hilo la ejecute a la vez.

In [3]:
contador = 0
cerrojo = threading.Lock()

def incrementa_seguro(veces):
    global contador
    for _ in range(veces):
        with cerrojo:
            actual = contador
            time.sleep(0)
            contador = actual + 1

hilos = [threading.Thread(target=incrementa_seguro, args=(1000,)) for _ in range(8)]
for hilo in hilos:
    hilo.start()
for hilo in hilos:
    hilo.join()
assert contador == 8000
print(f"esperado 8000, obtenido {contador}")

esperado 8000, obtenido 8000


## 4. Productor-consumidor con `queue`

Un productor mete 10 pedidos en una `queue.Queue` y dos consumidores los procesan. La cola ya es segura entre hilos y, con `maxsize`, frena al productor si va muy por delante. Al acabar, el productor mete un `None` por consumidor como señal de fin.

In [4]:
import queue

def productor(cola, pedidos, consumidores):
    for i in range(1, pedidos + 1):
        cola.put(f"pedido {i}")
        time.sleep(0.01)
    for _ in range(consumidores):
        cola.put(None)

def consumidor(nombre, cola, procesados):
    while (pedido := cola.get()) is not None:
        time.sleep(0.03)
        procesados.append((nombre, pedido))

cola = queue.Queue(maxsize=3)
procesados = []
hilos = [threading.Thread(target=productor, args=(cola, 10, 2))]
hilos += [threading.Thread(target=consumidor, args=(f"consumidor {c}", cola, procesados)) for c in "AB"]
for hilo in hilos:
    hilo.start()
for hilo in hilos:
    hilo.join()

for nombre, pedido in procesados:
    print(nombre, "->", pedido)
assert sorted(p for _, p in procesados) == sorted(f"pedido {i}" for i in range(1, 11))

consumidor A -> pedido 1
consumidor B -> pedido 2
consumidor A -> pedido 3
consumidor B -> pedido 4
consumidor A -> pedido 5
consumidor B -> pedido 6
consumidor A -> pedido 7
consumidor B -> pedido 8
consumidor A -> pedido 9
consumidor B -> pedido 10


## 5. `ThreadPoolExecutor` para E/S

Simular 10 descargas de 0,2 s con `time.sleep` y comparar hacerlas una tras otra con repartirlas en un `ThreadPoolExecutor`. `map` devuelve los resultados en orden; `as_completed` los da según van terminando.

In [5]:
import random
from concurrent.futures import ThreadPoolExecutor, as_completed

def descarga(url):
    time.sleep(0.2)
    return url, len(url) * 100

urls = [f"https://ejemplo.com/pagina{i}" for i in range(10)]

inicio = time.perf_counter()
secuencial = [descarga(url) for url in urls]
t_secuencial = time.perf_counter() - inicio

inicio = time.perf_counter()
with ThreadPoolExecutor(max_workers=10) as ejecutor:
    con_hilos = list(ejecutor.map(descarga, urls))
t_hilos = time.perf_counter() - inicio

assert secuencial == con_hilos
print(f"secuencial: {t_secuencial:.2f} s, con hilos: {t_hilos:.2f} s")

secuencial: 2.00 s, con hilos: 0.20 s


In [6]:
def descarga_variable(i):
    segundos = random.uniform(0.05, 0.3)
    time.sleep(segundos)
    return i, segundos

with ThreadPoolExecutor(max_workers=8) as ejecutor:
    futuros = [ejecutor.submit(descarga_variable, i) for i in range(8)]
    for futuro in as_completed(futuros):
        i, segundos = futuro.result()
        print(f"descarga {i} lista en {segundos:.2f} s")

descarga 0 lista en 0.05 s
descarga 2 lista en 0.06 s
descarga 1 lista en 0.07 s
descarga 4 lista en 0.08 s
descarga 6 lista en 0.17 s
descarga 7 lista en 0.18 s
descarga 3 lista en 0.18 s


descarga 5 lista en 0.21 s


## 6. El GIL: hilos con cálculo intensivo

En CPython el GIL (*Global Interpreter Lock*) solo deja que un hilo ejecute código Python en cada instante. Mientras un hilo espera E/S suelta el GIL y otro avanza, por eso los hilos aceleran las descargas; pero con cálculo puro se turnan y no hay paralelismo real. (Desde Python 3.13 existe una versión experimental sin GIL, *free-threaded*, que no es la habitual.)

Escribir en `tareas_cpu.py` una función que cuenta primos en un tramo y comprobar que repartir los tramos entre hilos no acelera el cálculo.

In [7]:
%%writefile tareas_cpu.py
def cuenta_primos(tramo):
    inicio, fin = tramo
    total = 0
    for n in range(max(inicio, 2), fin):
        if all(n % d for d in range(2, int(n ** 0.5) + 1)):
            total += 1
    return total

Writing tareas_cpu.py


In [8]:
import os
import sys

sys.dont_write_bytecode = True  # sin __pycache__ junto al notebook
sys.path.insert(0, os.getcwd())
from tareas_cpu import cuenta_primos

LIMITE = 800_000
tramos = [(i * LIMITE // 8, (i + 1) * LIMITE // 8) for i in range(8)]

inicio = time.perf_counter()
primos = sum(map(cuenta_primos, tramos))
t_secuencial = time.perf_counter() - inicio

inicio = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as ejecutor:
    primos_hilos = sum(ejecutor.map(cuenta_primos, tramos))
t_hilos = time.perf_counter() - inicio

assert primos == primos_hilos
print(f"{primos} primos menores que {LIMITE}")
print(f"secuencial: {t_secuencial:.2f} s, 4 hilos: {t_hilos:.2f} s")

63951 primos menores que 800000
secuencial: 1.77 s, 4 hilos: 1.88 s


## 7. `ProcessPoolExecutor`

Repartir los mismos tramos entre 4 procesos, cada uno con su propio intérprete y su propio GIL. En Windows los procesos hijos se crean con *spawn*: arrancan un intérprete nuevo que importa la función, así que esta tiene que estar en un módulo importable (`tareas_cpu.py`) y no definida en el notebook.

In [9]:
import multiprocessing
from concurrent.futures import ProcessPoolExecutor

print("método de arranque:", multiprocessing.get_start_method(), "| núcleos:", os.cpu_count())

inicio = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as ejecutor:
    primos_procesos = sum(ejecutor.map(cuenta_primos, tramos))
t_procesos = time.perf_counter() - inicio

assert primos_procesos == primos
print(f"secuencial: {t_secuencial:.2f} s, 4 procesos: {t_procesos:.2f} s")

método de arranque: spawn | núcleos: 16


secuencial: 1.77 s, 4 procesos: 0.79 s


## 8. `multiprocessing.Pool` y comparativa

Hacer lo mismo con `multiprocessing.Pool` y comparar los tiempos de las cuatro formas. Después repetirlo con tramos diminutos: arrancar procesos y enviarles los datos cuesta tiempo, así que los procesos solo compensan si cada tarea es lo bastante larga.

In [10]:
inicio = time.perf_counter()
with multiprocessing.Pool(processes=4) as pool:
    primos_pool = sum(pool.map(cuenta_primos, tramos))
t_pool = time.perf_counter() - inicio
assert primos_pool == primos

for nombre, t in [("secuencial", t_secuencial), ("4 hilos", t_hilos),
                  ("ProcessPoolExecutor", t_procesos), ("multiprocessing.Pool", t_pool)]:
    print(f"{nombre:<22}{t:6.2f} s   x{t_secuencial / t:.1f}")

secuencial              1.77 s   x1.0
4 hilos                 1.88 s   x0.9
ProcessPoolExecutor     0.79 s   x2.2
multiprocessing.Pool    0.75 s   x2.4


In [11]:
pequenos = [(i * 100, (i + 1) * 100) for i in range(8)]

inicio = time.perf_counter()
sum(map(cuenta_primos, pequenos))
t_sec = time.perf_counter() - inicio

inicio = time.perf_counter()
with multiprocessing.Pool(processes=4) as pool:
    sum(pool.map(cuenta_primos, pequenos))
t_proc = time.perf_counter() - inicio
print(f"tramos pequeños -> secuencial: {t_sec * 1000:.2f} ms, procesos: {t_proc * 1000:.0f} ms")

tramos pequeños -> secuencial: 0.33 ms, procesos: 90 ms


## 9. `asyncio.gather` frente a hilos

Simular 100 peticiones de 0,1 s con corrutinas (`asyncio.sleep`) lanzadas a la vez con `asyncio.gather`, y compararlo con 100 hilos. Las corrutinas se turnan en un único hilo cada vez que hacen `await`. En Jupyter ya hay un bucle de eventos en marcha, así que se usa `await` directamente en la celda (en un script sería `asyncio.run(main())`).

In [12]:
import asyncio

async def peticion_async(i):
    await asyncio.sleep(0.1)
    return i * 2

def peticion_hilo(i):
    time.sleep(0.1)
    return i * 2

inicio = time.perf_counter()
resultados = await asyncio.gather(*(peticion_async(i) for i in range(100)))
t_async = time.perf_counter() - inicio

inicio = time.perf_counter()
with ThreadPoolExecutor(max_workers=100) as ejecutor:
    resultados_hilos = list(ejecutor.map(peticion_hilo, range(100)))
t_hilos = time.perf_counter() - inicio

assert resultados == resultados_hilos
print(f"asyncio (1 hilo): {t_async:.2f} s, 100 hilos: {t_hilos:.2f} s")

asyncio (1 hilo): 0.12 s, 100 hilos: 0.11 s


Una llamada bloqueante (`time.sleep`) dentro de una corrutina para todo el bucle de eventos y las tareas acaban yendo de una en una. Para usar código bloqueante desde `asyncio` se manda a un hilo con `asyncio.to_thread`.

In [13]:
async def bloqueante(i):
    time.sleep(0.1)
    return i * 2

async def con_to_thread(i):
    return await asyncio.to_thread(peticion_hilo, i)

for corrutina in (peticion_async, bloqueante, con_to_thread):
    inicio = time.perf_counter()
    await asyncio.gather(*(corrutina(i) for i in range(10)))
    print(f"{corrutina.__name__:<15}{time.perf_counter() - inicio:.2f} s")

peticion_async 0.11 s


bloqueante     1.00 s
con_to_thread  0.10 s
